# Exercise 3: Fitting

## Analysis of Exponential Decay Data

You are given a dataset representing exponential decay with added noise. The fraction of remaining particles follows the equation:  


$$y = e^{t/\tau}$$


where:  
- $ \tau $ is the decay constant,  
- $ t $ represents time,  
- measurement uncertainty is estimated to be $ \sigma = 0.1 $ for all times.  

**Tasks**  

**1. Compute the Chi-Squared Statistic**  
- Evaluate $ \chi^2 $ for $ \tau $ values between 1 and 2.  
- Identify the best-fit $ \tau $ by finding the minimum $ \chi^2 $.  

**2. Estimate Uncertainty**  
- Use the $ \Delta \chi^2 $ method to determine the confidence interval for $ \tau $.  

**3. Visualize the Data and Best Fit**  
- Plot the given data with error bars.  
- Overlay the best-fit exponential decay curve.  

**4. Graphical Representation of $ \chi^2 $**  
- Plot $ \chi^2 $ as a function of $ \tau $.  
- Highlight the $ 1\sigma $ confidence regions based on $ \Delta \chi^2 $.  

**5. Goodness of fit**
- Looking at the mimimum $\chi^2$, do you think the uncertainty estimation is realistic?

In [3]:
import numpy as np
t, data = np.loadtxt("data_ex1.txt", unpack=True) #here data refers to the fraction of remaining particles (according to the description of the exercise)
data_errors = np.full_like(data, 0.1)

In the lecture, we learned how to compute the best estimate of a parameter analytically in the case of a linear function. Now, compute the best estimate for $\tau$, including the uncertainty, in a similar way. 

Do you get "compatible" results ?

**Hint**: You may want to use a coordinate transformation.

In [69]:
# 1.
def compute_chisquared(x,y,var_y,model,**dict_args):
    """Computes the chi-squared of the set of data
    
    The function needs the data as inputs, the variance of the y data and the function of the distribution to
    which the data belongs. It returns the not-normalized chi-squared. Additionally you can give more inputs:
    - model_param: the collection of parameters needed by the function  model
    - var_x:       the variance of the x coordinates"""
    import numpy as np

    # Checking basic input sizes
    if (len(x) != len(y)) or (len(x) != len(var_y)):
        print("Inputs to compute_chisquared have different lengths. Aborting")
        return 0
    N_points = len(x)

    # Managing variances
    var = np.zeros(N)
    i = 0
    if 'err_x' in dict_args:
        if len(x) != len(var_x):
            print("Inputs to compute_chisquared have different lengths. Aborting")
            return 0

        from scipy.optimize import approx_fprime
        while i < N:
            var[i] = var_x[i]*(approx_fprime(x[i],model,1e-8,dict_args['model_param']))**2 + var_y[i]
            i += 1
    else:
        var[i] = var_y[i]

    # Computing chi-squared
    chi_squared = 0
    i = 0
    while i < N:
        chi_squared += (model(x[i],dict_args['model_param']) - y[i])**2/var[i]
        i += 1

    # Returning the final outcome
    return chi_squared
    

In [70]:
# 1.
def compute_chisquared(x,y,var_y,model,**dict_args):
    """Computes the chi-squared of the set of data
    
    The function needs the data as inputs, the variance of the y data and the function of the distribution to
    which the data belongs. It returns the not-normalized chi-squared. Additionally you can give more inputs:
    - model_param: the collection of parameters needed by the function  model
    - var_x:       the variance of the x coordinates"""
    import numpy as np

    # Checking basic input sizes
    if (len(x) != len(y)) or (len(x) != len(var_y)):
        print("Inputs to compute_chisquared have different lengths. Aborting")
        return 0
    N_points = len(x)

    # Managing variances
    var = np.zeros(N)
    i = 0
    try:
        if len(x) != len(dict_args['var_x']):
            print("Inputs to compute_chisquared have different lengths. Aborting")
            return 0
        from scipy.optimize import approx_fprime
        try:
            while i < N:
                var[i] = dict_args['var_x'][i]*(approx_fprime(x[i],model,1e-8,dict_args['model_param']))**2 + var_y[i]
                i += 1
        except KeyError:
            while i < N:
                var[i] = dict_args['var_x'][i]*(approx_fprime(x[i],model,1e-8))**2 + var_y[i]
                i += 1
        print("compute_chisquared: var_x detected!")
    except KeyError:
        while i < N:
            var[i] = var_y[i]
        print("compute_chisquared: no var_x detected!")

    # Computing chi-squared
    chi_squared = 0
    i = 0
    try:
        while i < N:
            chi_squared += (model(x[i],dict_args['model_param']) - y[i])**2/var[i]
            i += 1
        print("compute_chisquared: model_param detected!")
    except KeyError:
        while i < N:
            chi_squared += (model(x[i]) - y[i])**2/var[i]
            i += 1
        print("compute_chisquared: no model_param detected!")        

    # Returning the final outcome
    return chi_squared
    

In [66]:
from scipy.optimize import approx_fprime

#help(approx_fprime)

In [65]:
def exponential(x,*tau):
    import numpy as np
    print(tau)
    return np.exp(-x/tau)
    

from scipy.optimize import approx_fprime
approx_fprime(2,exponential,1e-8,4)


(4,)
(4,)


array([-0.15163266])

## Polynomial Fit with Parameter Uncertainty

You are given data that was generated using a polynomial of degree 5. Therefore, you have 6 free parameters (the 6 coefficients). Your task is to perform a fit to obtain these parameters and their covariances.

**Steps:**
1. Fit a polynomial to the data and determine the parameters.
2. Calculate the uncertainties for each parameter.
3. **Bonus**: Instead of just showing the 1D uncertainties for each parameter, visualize the correlations between these uncertainties.

You can use an optimizer from `scipy` to perform the fit and "TriangleChain" for the last point.

In [6]:
x, data = np.loadtxt("data_ex2.txt", unpack=True)

In [ ]:
!pip install trianglechain

## Goodness of fit

You are given datasets $x$ and $y$ that follow a polynomial relationship, with the uncertainty in $y$ equal to 5. Fit both a first-degree (linear) and a second-degree (quadratic) polynomial to the data. Compute the reduced chi-squared and the p-value for each model. Determine which model provides a better fit based on these statistical measures.

What would happen to the p value if you further increase the degrees of the polynomial?

In [9]:
x, y = np.loadtxt("data_ex3.txt", unpack=True)
y_err = np.ones_like(y) * 5

## Gradient descent
Minimize the function:

$$
f(x, y) = x^2 + (y - 1)^2 + 0.1 \cdot x^4 + 0.1 \cdot y^4
$$

1. **Compute the Gradient**  
   Find the partial derivatives of the function with respect to $x$ and $y$.

2. **Implement Gradient Descent**  
   Update the parameters using:
   $$
   x_{\text{new}} = x_{\text{old}} - \alpha \cdot \frac{\partial f}{\partial x}, \quad y_{\text{new}} = y_{\text{old}} - \alpha \cdot \frac{\partial f}{\partial y}
   $$

3. **Plot the Path to the Minimum**  
   Run gradient descent with different starting points and learning rates. Plot the optimization path to the minimum for the different scenarios and plot how the step size evolves.

4. **Different function**
    Change the initial function (e.g. change $x^2$ to $x^3$) and redo Steps 1-3. Do you have to adapt the learning rate?